# Bronze Layer: Raw JSON Ingestion

This notebook is the **bronze** stage of the NEM medallion pipeline. It reads the three raw JSON files already saved to a Unity Catalog Volume and writes each one, unnested but otherwise untouched, into its own bronze Delta table.

Deliberately out of scope here: type casting, null handling, deduplication, derived columns, and any exploratory analysis. Every column is written as a string. All of that cleanup happens downstream in the silver layer.

Outputs:
- `nem_project.`1_bronze`.facilities`
- `nem_project.`1_bronze`.facility_power_emissions`
- `nem_project.`1_bronze`.region_price_demand`

## 0. Setup

In [0]:
#ijson lets us stream-parse the large power/emissions file one facility at a time,
#instead of loading the whole ~1GB file into memory with json.load().
#%pip install ijson

In [0]:
'Import libraries'

import json
import ijson
import pandas as pd

from pyspark.sql.functions import col
from pyspark.sql.types import StringType

In [0]:
'Config: where the raw JSON lives, and where the bronze tables get written'

#Path to the Unity Catalog Volume holding the raw JSON files.
#Adjust this to match wherever the three files were actually uploaded.
VOLUME_PATH = "/Volumes/nem_project/0-raw_data/api_json_files"

path_facilities = f"{VOLUME_PATH}/facilities.json"
path_power_emissions = f"{VOLUME_PATH}/facility_power_emissions_sep2025_aug2026.json"
path_region_price_demand = f"{VOLUME_PATH}/region_price_demand_sep2025_aug2026.json"

table_facilities = "nem_project.`1_bronze`.facilities"
table_power_emissions = "nem_project.`1_bronze`.facility_power_emissions"
table_region_price_demand = "nem_project.`1_bronze`.region_price_demand"

#How many facilities to accumulate per batch when streaming the large power/emissions file.
BATCH_SIZE = 25

In [0]:
'Helper function: Cast every column of a Spark DataFrame to StringType to avoid datatype mismatch'

def cast_all_columns_to_string(spark_df):
    """Bronze tables store raw values as strings only; typed casting is a silver-layer concern. Casting via Spark (rather than pandas .astype(str)) keeps real nulls as nulls instead of turning them into the literal text 'None'."""
    return spark_df.select([col(c).cast(StringType()).alias(c) for c in spark_df.columns])

## 1. Facilities → `nem_project.bronze.facilities`

`facilities.json` is a single JSON object with a top-level `data` list of facilities, each carrying a nested `location` dict and a nested `units` list. Small file, so it's loaded and flattened in one pass. Every facility is expanded to one row per unit (a facility with 3 units becomes 3 rows).

In [0]:
'Load facilities.json'

with open(path_facilities, "r") as f:
    facilities_payload = json.load(f)

facility_records = facilities_payload["data"]
print(f"Facilities in file: {len(facility_records)}")

In [0]:
'Flatten facilities + their units into one row per unit'

flat_rows = []

for facility in facility_records:
    #Fields shared by every unit belonging to this facility.
    facility_info = {
        "facility_code": facility.get("code"),
        "facility_name": facility.get("name"),
        "network_id": facility.get("network_id"),
        "network_region": facility.get("network_region"),
        "lat": facility["location"]["lat"] if facility.get("location") else None,
        "lng": facility["location"]["lng"] if facility.get("location") else None,
        "facility_description": facility.get("description"),
    }

    #One output row per unit, unit fields layered on top of the shared facility fields.
    for unit in facility.get("units", []):
        row = facility_info.copy()
        row.update({
            "unit_code": unit.get("code"),
            "fueltech_id": unit.get("fueltech_id"),
            "status_id": unit.get("status_id"),
            "capacity_registered": unit.get("capacity_registered"),
            "capacity_maximum": unit.get("capacity_maximum"),
            "capacity_storage": unit.get("capacity_storage"),
            "data_first_seen": unit.get("data_first_seen"),
            "data_last_seen": unit.get("data_last_seen"),
            "dispatch_type": unit.get("dispatch_type"),
        })
        flat_rows.append(row)

df_facilities = pd.DataFrame(flat_rows)
print(f"Flattened rows (facility x unit): {len(df_facilities)}")
df_facilities.head()

In [0]:
"Create Delta table"
#'pandas -> Spark -> Delta bridge'
#this code takes a table we already built in pandas, converts everything to text, saves it as a proper named Delta table in Databrick.
#Spark's only job here is to cast every column to string and persist it as a managed Delta table under its three-level UC name.

sdf_facilities = spark.createDataFrame(df_facilities)
sdf_facilities = cast_all_columns_to_string(sdf_facilities)

sdf_facilities.write.format("delta").mode("overwrite").saveAsTable(table_facilities)

print(f"Wrote {sdf_facilities.count()} rows to {table_facilities}")

## 2. Facility Power & Emissions → `nem_project.bronze.facility_power_emissions`

`facility_power_emissions_sep2025_aug2026.json` is a single JSON object keyed by `facility_code`. Each facility's value holds a `data` list containing separate blocks for the `power` and `emissions` metrics, each with per-unit `results` and a `[timestamp, value]` time series.

At roughly 1GB / ~40M rows, this file is too large to `json.load()` in one call. Instead we stream it with `ijson.kvitems`, which yields one `(facility_code, facility_data)` pair at a time without ever holding the full file in memory. Facilities are accumulated into batches of `BATCH_SIZE`, flattened, and appended to the Delta table batch by batch, so peak memory stays bounded regardless of file size.

In [0]:
'Flatten one batch of facilities into power/emissions rows'

def flatten_power_emissions_batch(facility_batch):
    """facility_batch is a list of (facility_code, facility_data) pairs.
    Build a (unit_code, timestamp) -> value map per metric, then union the keys so a
    single output row carries both power and emissions for that unit/timestamp."""
    records = []

    for facility_code, facility_data in facility_batch:
        data_blocks = facility_data.get("data", [])

        power_blocks = [b for b in data_blocks if b.get("metric") == "power"]
        emissions_blocks = [b for b in data_blocks if b.get("metric") == "emissions"]

        power_dict = {}
        for block in power_blocks:
            for result in block.get("results", []):
                unit_code = result["columns"].get("unit_code")
                for ts, val in result.get("data", []):
                    power_dict.setdefault((unit_code, ts), val)

        emissions_dict = {}
        for block in emissions_blocks:
            for result in block.get("results", []):
                unit_code = result["columns"].get("unit_code")
                for ts, val in result.get("data", []):
                    emissions_dict.setdefault((unit_code, ts), val)

        all_keys = set(power_dict.keys()) | set(emissions_dict.keys())
        for unit_code, ts in all_keys:
            records.append({
                "facility_code": facility_code,
                "unit_code": unit_code,
                "time": ts,
                "power": power_dict.get((unit_code, ts)),
                "emissions": emissions_dict.get((unit_code, ts)),
            })

    return records

In [0]:
'Stream-parse the large file and write it batch by batch'

def write_batch(records, batch_num):
    """First batch overwrites the table (fresh run); every batch after that appends,
    so the table ends up holding every facility's rows without ever materialising the
    whole dataset in memory at once."""
    pdf = pd.DataFrame(records)
    sdf = spark.createDataFrame(pdf)
    sdf = cast_all_columns_to_string(sdf)

    write_mode = "overwrite" if batch_num == 0 else "append"
    sdf.write.format("delta").mode(write_mode).saveAsTable(table_power_emissions)

    print(f"Batch {batch_num}: wrote {len(records)} rows (mode={write_mode})")


facility_batch = []
batch_num = 0

with open(path_power_emissions, "rb") as f:
    #kvitems(f, "") streams top-level key/value pairs of the root JSON object one at a
    #time, so only one facility's data is ever fully materialised in memory at once.
    for facility_code, facility_data in ijson.kvitems(f, ""):
        facility_batch.append((facility_code, facility_data))

        if len(facility_batch) >= BATCH_SIZE:
            batch_records = flatten_power_emissions_batch(facility_batch)
            write_batch(batch_records, batch_num)

            facility_batch = []
            batch_num += 1

#Flush the final, possibly partial, batch.
if facility_batch:
    batch_records = flatten_power_emissions_batch(facility_batch)
    write_batch(batch_records, batch_num)

print(f"Finished writing {table_power_emissions}")

## 3. Region Price & Demand → `nem_project.bronze.region_price_demand`

`region_price_demand_sep2025_aug2026.json` is a single JSON object keyed by `network_region`. Each region's value holds a `data` list with separate blocks for the `price` and `demand` metrics, each carrying a `[timestamp, value]` time series (minus the date/time/timezone splitting, which is a derived-column step left for silver). Small file, loaded and flattened in one pass.

In [0]:
'Load region_price_demand JSON'

with open(path_region_price_demand, "r") as f:
    region_payload = json.load(f)

print(f"Regions in file: {len(region_payload)}")

In [0]:
'Flatten region price/demand into one row per (region, timestamp)'

region_records = []

for region_code, region_data in region_payload.items():
    data_blocks = region_data.get("data", [])

    price_blocks = [b for b in data_blocks if b.get("metric") == "price"]
    demand_blocks = [b for b in data_blocks if b.get("metric") == "demand"]

    price_dict = {}
    for block in price_blocks:
        for result in block.get("results", []):
            for ts, val in result.get("data", []):
                price_dict.setdefault(ts, val)

    demand_dict = {}
    for block in demand_blocks:
        for result in block.get("results", []):
            for ts, val in result.get("data", []):
                demand_dict.setdefault(ts, val)

    all_ts = set(price_dict.keys()) | set(demand_dict.keys())
    for ts in all_ts:
        region_records.append({
            "network_region": region_code,
            "time": ts,
            "price": price_dict.get(ts),
            "demand": demand_dict.get(ts),
        })

df_region_price_demand = pd.DataFrame(region_records)
print(f"Flattened rows (region x timestamp): {len(df_region_price_demand)}")
df_region_price_demand.head()

In [0]:
"Create Delta tables"
#pandas -> Spark -> Delta bridge
#Same bridge as the facilities table: pandas did the flattening, Spark just casts

#every column to string and writes it out as a managed Delta table.
sdf_region_price_demand = spark.createDataFrame(df_region_price_demand)
sdf_region_price_demand = cast_all_columns_to_string(sdf_region_price_demand)

sdf_region_price_demand.write.format("delta").mode("overwrite").saveAsTable(table_region_price_demand)

print(f"Wrote {sdf_region_price_demand.count()} rows to {table_region_price_demand}")